In [1]:
import sys
sys.path.append("..")
import torch
from transformer_lens import HookedTransformer
from src.data import PROJECT_ROOT
from datasets import load_from_disk
from src.data import load_model_and_tokenizer, load_rultaker, prepare_example, evaluate_model_capability

# Overall smoke test

In [2]:
# Проверка CUDA
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
# Загрузка модели 410m
model, tokenizer = load_model_and_tokenizer(model_size="410m")
print("Модель загружена, устройство:", next(model.parameters()).device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model EleutherAI/pythia-410m-deduped into HookedTransformer
Модель загружена, устройство: cuda:0


In [4]:
# Проверка кэширования
tokens = model.to_tokens("The cat is")
logits, cache = model.run_with_cache(tokens)
print("Доступные хуки:", list(cache.keys())[:5], "...")

Доступные хуки: ['hook_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q'] ...


In [6]:
# Загрузка датасета (depth-2)
ds = load_rultaker(split="train", variant="depth-2", max_samples=5)
print("Пример датасета:", ds[0].keys())
print("Первый пример:", ds[0]["english"]["theory_statements"][:2], "->", ds[0]["english"]["assertion_statement"], "label:", ds[0]["theory_assertion_instance"]["label"])

Пример датасета: dict_keys(['json_class', 'id', 'theory_assertion_instance', 'logical_forms', 'english', 'logic_program'])
Первый пример: ['The bald eagle is nice.', 'The squirrel visits the bald eagle.'] -> The squirrel visits the bald eagle. label: True


# Model sanity check

In [3]:
PROMPTS = {
    "prompt_base": "{text} The assertion is ",
    "qwen_instruct": "<|im_start|>user\n{text}\nIs the assertion true or false? Answer with 'True' or 'False'.<|im_end|>\n<|im_start|>assistant\n"
}

PROMPT_TF = "Is the following assertion true or false? Answer only with 'True' or 'False'.\n\n{text}\nAnswer:"
PROMPT_YN = "Is the following assertion correct? Answer only with 'Yes' or 'No'.\n\n{text}\nAnswer:"

In [4]:
VARIANT = "depth-0"
cache_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}_small"
dataset = load_from_disk(str(cache_path))

In [5]:
# Если в кэше отсутствует поле text, применяем унификацию
if "text" not in dataset["dev"].column_names:
    dataset = dataset.map(prepare_example, remove_columns=dataset.column_names)

## Проверка Pythia-1b (base-модель)

In [6]:
model_pythia, _ = load_model_and_tokenizer(model_size="1b", device="cuda", dtype=torch.float16)

KeyboardInterrupt: 

In [ ]:
acc_pythia = evaluate_model_capability(model_pythia, dataset["dev"], PROMPTS["prompt_base"], batch_size=32)
print(f"Pythia-1b Dev Accuracy: {acc_pythia:.4f}\n")

Evaluating capability: 100%|██████████| 16/16 [00:10<00:00,  1.52it/s]

Pythia-1b Dev Accuracy: 0.5200



In [8]:
del model_pythia
torch.cuda.empty_cache()

## Проверка Qwen2.5-1.5b (instruct)

In [ ]:
model_qwen, _ = load_model_and_tokenizer(model_size="qwen1.5b_instruct", device="cuda", dtype=torch.bfloat16)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model Qwen/Qwen2.5-1.5B-Instruct into HookedTransformer


In [10]:
acc_qwen = evaluate_model_capability(model_qwen, dataset["dev"], PROMPTS["qwen_instruct"], batch_size=32)
print(f"Qwen Dev Accuracy (small): {acc_qwen:.4f}")

Evaluating capability: 100%|██████████| 16/16 [01:23<00:00,  5.21s/it]

Qwen Dev Accuracy (small): 0.5640


In [8]:
acc_tf = evaluate_model_capability(model_qwen, dataset["dev"], PROMPT_TF, batch_size=16)
print(f"Accuracy (True/False): {acc_tf:.4f}")

Evaluating capability: 100%|██████████| 32/32 [00:19<00:00,  1.67it/s]

Accuracy (True/False): 0.4840


In [9]:
acc_yn = evaluate_model_capability(model_qwen, dataset["dev"], PROMPT_YN, batch_size=16)
print(f"Accuracy (Yes/No): {acc_yn:.4f}")

Evaluating capability: 100%|██████████| 32/32 [00:16<00:00,  1.91it/s]

Accuracy (Yes/No): 0.5020


In [11]:
example = dataset["dev"][0]
text = PROMPT_TF.format(text=example["text"])
tokens = model_qwen.to_tokens([text], prepend_bos=True).to("cuda")

In [13]:
with torch.no_grad():
    logits = model_qwen(tokens)[0, -1, :]

In [14]:
for word in ["True", "False", "Yes", "No"]:
    ids = model_qwen.tokenizer.encode(word, add_special_tokens=False)
    if ids:
        tid = ids[-1]
        print(f"  {word:5s} (ID {tid:>5}): {logits[tid].item():+.3f}")

  True  (ID  2514): +14.000
  False (ID  4049): +15.500
  Yes   (ID  9454): +5.938
  No    (ID  2753): +4.812


In [1]:
del model_pythia
torch.cuda.empty_cache()

NameError: name 'model_pythia' is not defined

## Проверка Qwen2.5-1.5b (base)

In [6]:
model_qwen_base, _ = load_model_and_tokenizer(model_size="qwen1.5b_base", device="cuda", dtype=torch.bfloat16)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\avmar\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-1.5B into HookedTransformer


In [7]:
acc_base = evaluate_model_capability(model_qwen_base, dataset["dev"], PROMPTS["prompt_base"], batch_size=32)
print(f"Qwen2.5-1.5B-Base Dev Accuracy: {acc_base:.4f}")

Evaluating capability: 100%|██████████| 16/16 [01:02<00:00,  3.90s/it]

Qwen2.5-1.5B-Base Dev Accuracy: 0.5060
